### Runnable Config 
So far, you've learned to create and execute Runnables using methods such as .invoke(), .batch(), and .stream().
But suppose you want to control how a Runnable executes without changing its core logic.
For example, you might want to:
- Add tags to identify a particular execution.
- Attach metadata, such as a project name.
- Control concurrency when processing multiple inputs.
- Pass configuration information through a chain.
This is where RunnableConfig comes in.
Think of a Runnable as a worker and its configuration as a set of instructions about how that worker should operate.

In [5]:
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableLambda,RunnableConfig
from dotenv import load_dotenv

load_dotenv()

def to_uppercase(text:str):
    return text.upper()

runnable = RunnableLambda(to_uppercase)

response = runnable.invoke("learning langchain",config={
    "tags": ["practice"],
    "metadata":{"topic":"RunnableConfig"}
})

print(response)

LEARNING LANGCHAIN


In [8]:
config = RunnableConfig(
    tags=["practice"],
    metadata={"topic":"RunnableConfig"}
)

response = runnable.invoke("learning AI",config=config)

print(response)
print(config)

LEARNING AI
{'tags': ['practice'], 'metadata': {'topic': 'RunnableConfig'}}


#### In a chain the runnables config is applied to both the runnables mentioned 

Modify your chain so that it contains:  
1. A RunnableLambda that converts text to uppercase.  
2. A second RunnableLambda that adds the prefix "RESULT: " to the text.  
Invoke the chain with the input "learning langchain" and the same tags and metadata configuration.  
Expected output:  
RESULT: LEARNING LANGCHAIN  

In [4]:
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableLambda
from dotenv import load_dotenv

load_dotenv()

def to_uppercase(text:str):
    return text.upper()

def add_prefix(text:str):
    return f"RESULT: {text}"

chain = RunnableLambda(to_uppercase) | RunnableLambda(add_prefix)

response = chain.invoke(
    "learning langchain",
    config = {
        "tags":["practice"],
        "metadata":{
            "topic":"RunnableConfig"
        }
    }
)

print(response)

RESULT: LEARNING LANGCHAIN


### Configurable Fields
re-configure the initially defined fields

### Standard Way - best practice

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableLambda,ConfigurableField,RunnableConfig
from dotenv import load_dotenv

load_dotenv()

llm = init_chat_model(
    model="groq:qwen/qwen3.8-27b",
    max_tokens = 500,
    temperature = 0
)

# expose temperature as configurable knob
llm = llm.configurable_fields(temperature = ConfigurableField(id="temperature"))


response = llm.invoke("Tell me a short joke on corporate people")
print("Using default temperature : 0\n",response.content)

config = RunnableConfig(configurable={"temperature":0.8})

response_config = llm.invoke("Tell me a short joke on corporate people",config=config)
print("\nUsing Configurable temperature: 0.8\n",response_config.content)

Using default temperature : 0
 Why did the corporate employee bring a ladder to the meeting?

Because they heard the discussion was going to be **high-level**.
Using Configurable temperature: 0.8
 Why did the corporate manager bring a ladder to the meeting?

Because everything was below his pay grade.


### Quick way

In [9]:
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableLambda
from dotenv import load_dotenv

load_dotenv()

llm = init_chat_model(
    model="groq:qwen/qwen3.8-27b",
    max_tokens = 500,
    temperature = 0
)

response = llm.invoke(
    "Tell me a short joke on corporate people",
    config = {
        "configurable":{"temperature":0.8}
    }
)

print(response.content)

Why did the corporate employee bring a ladder to the meeting?

Because they heard the discussion was going to be **high-level**.


### Configurable alternatives
register another configurable, if we didnt explicitly pass the the field in the config field

Create two RunnableLambda objects:
- model_a returns "Response from Model A"
- model_b returns "Response from Model B"
Then:
1. Use configurable_alternatives() to register model_b as an alternative to model_a.
2. Set the ConfigurableField ID to "model_choice".
3. Set the default key to "model_a".
4. Invoke the configurable Runnable once without a configuration override.
5. Invoke it again with config={"configurable": {"model_choice": "model_b"}}.

In [27]:
from langchain_core.runnables import RunnableLambda, ConfigurableField

def response_a(data):
    return "Response from Model A"

def response_b(data):
    return "Response from Model B"

model_a = RunnableLambda(response_a) | RunnableLambda(lambda x:x)
model_b = RunnableLambda(response_b) | RunnableLambda(lambda x:x)

configurable_model = model_a.configurable_alternatives(
    ConfigurableField(id="model_choice"),
    default_key="model_a",
    model_b=model_b
)

default_response = configurable_model.invoke("test")
print(f"default by {configurable_model.default_key}:{default_response}")

config_res = configurable_model.invoke("test",config={"configurable":{"model_choice":"model_b"}})
print(f"configurable response by model_b: {config_res}")

default by model_a:Response from Model A
configurable response by model_b: Response from Model B


### Example 2- Runtime model selector
Scenario: You're building a LangChain application that can switch between two response styles without rewriting the chain.
- Model A: returns a short, simple explanation.
- Model B: returns a detailed explanation.
For now, use RunnableLambda to simulate the two models.

Your requirements:
1. Create two Runnables, model_a and model_b.
2. Use configurable_alternatives() with ConfigurableField(id="response_style").
3. Set "model_a" as the default.
4. Invoke once using the default.
5. Invoke again with "response_style": "model_b".

Expected behavior:  
Invocation  	    Expected result  
No override  	    Short explanation  
Select model_b	Detailed explanation

In [5]:
from langchain_core.runnables import RunnableLambda, ConfigurableField

def short_resp(query):
    return "Short explanation"

def long_resp(query):
    return "Detailed explanation"

model_A = RunnableLambda(short_resp) | RunnableLambda(lambda x:x)
model_B = RunnableLambda(long_resp) | RunnableLambda(lambda x:x)

config_model = model_A.configurable_alternatives(ConfigurableField(id="response_style"),default_key = "model_A",model_B=model_B)

response_a = config_model.invoke("test")
print("No override",response_a)

response_b = config_model.invoke("test",config={"configurable":{"response_sytle":"model_B"}})
print("Select model_B",response_b)

No override Short explanation
Select model_B Short explanation


### using actual model

The overall structure will be:
1. Initialize two chat models using init_chat_model().
2. Register the second model as an alternative to the first.
3. Connect the configurable model to a ChatPromptTemplate.
4. Invoke the same chain with different model selections.
5. Compare the responses.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.runnables import ConfigurableField
from langchain_core.prompts import ChatPromptTemplate

llm1 = init_chat_model(model="groq:qwen/qwen3.8-27b",max_tokens=500)
llm2 = init_chat_model(model="groq:openai/gpt-oss-120b",max_tokens=500)

config_model = llm1.configurable_alternatives(ConfigurableField(id="model_choice"),default_key="llm1",llm2=llm2)

prompt = ChatPromptTemplate.from_template("Write a quote in 2 line on {topic}")

chain = prompt | config_model 

response_1 = chain.invoke({"topic":"Life"})
print(f"Quote-1: <from {response_1.response_metadata['model_name']}>\n{response_1.content}")

response_2 = chain.invoke({"topic":"Human"},config={"configurable":{"model_choice":"llm2"}})
print(f"Quote-2: <from {response_2.response_metadata['model_name']}>\n{response_2.content}")


Quote-1: <from qwen/qwen3.8-27b >
Life is not about waiting for the storm to pass,
but learning to dance in the rain.)
Quote-1: <from openai/gpt-oss-120b>
“Humanity is a single heartbeat echoing through countless lives,  
each pulse a reminder that we are both the question and the answer.”)
